# M11 · Fine-tuning e alinhamento: prática

1. Decidir com números: API ou modelo próprio?
2. LoRA à mão em PyTorch.
3. Chat template e máscara da perda.
4. Gerar dados por destilação.
5. Linhas de base: modelo pequeno sem treino e modelo grande.
6. SFT com QLoRA e avaliação.
7. DPO com os erros do próprio modelo.
8. GRPO: escrever uma recompensa verificável.
9. Esquecimento, fusão e publicação.

**Requisitos:** GPU com 16 GB ou mais (uma T4 do Google Colab chega) para as secções 5 a 9; `uv sync --extra finetuning`; `ANTHROPIC_API_KEY` para gerar dados e para a linha de base do modelo grande. As secções 1 a 3 correm em CPU.

No Colab: *Runtime → Change runtime type → T4 GPU*, clona o repositório e instala as dependências com a célula abaixo.

Lição de apoio: [licao.html](licao.html)

In [ ]:
# No Colab, descomenta:
# !git clone https://github.com/paulodiasfaria/ai-course.git && cd ai-course/modulos/11-fine-tuning-e-alinhamento
# !pip install -q "trl>=0.20" peft bitsandbytes datasets accelerate anthropic python-dotenv
import json, os, subprocess, sys
from pathlib import Path
from dotenv import load_dotenv
import torch

load_dotenv()
UC = Path("use-case").resolve()
sys.path.insert(0, str(UC))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (secções 1 a 3 apenas)")

def correr(*args):
    """Corre um script do use case e mostra o output à medida que sai."""
    p = subprocess.Popen([sys.executable, *args], cwd=UC, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for linha in p.stdout:
        print(linha, end="")
    if p.wait():
        raise RuntimeError(f"{args[0]} terminou com erro")

## 1. Decidir com números

A Loja Lusa recebe 1 milhão de reclamações por mês. Cada pedido tem ~350 tokens de entrada (prompt + reclamação) e ~60 de saída. Compara a API com uma GPU alugada. Ajusta os valores à tua realidade.

In [ ]:
pedidos_mes = 1_000_000
tok_in, tok_out = 350, 60
api = {"claude-haiku-5-5": (0.10, 0.50), "claude-sonnet-5-5": (2.0, 10.0)}   # USD por milhão de tokens
for m, (pi, po) in api.items():
    print(f"{m:<20} US$ {pedidos_mes * (tok_in * pi + tok_out * po) / 1e6:>8,.0f} por mês")
gpu_hora, pedidos_por_seg = 0.80, 40     # GPU de 24 GB alugada; vLLM com um modelo de 1,5B (módulo 12)
horas = pedidos_mes / pedidos_por_seg / 3600
print(f"{'GPU própria':<20} US$ {max(horas, 24 * 30 * 0.2) * gpu_hora:>8,.0f} por mês ({horas:.0f} h de cálculo; assumimos a GPU ligada pelo menos 20% do tempo)")

Com um modelo pequeno e barato na API, a poupança em dinheiro pode não justificar o trabalho. O fine-tuning ganha com volume muito alto, requisitos de privacidade, latência ou funcionamento offline. Faz sempre estas contas antes de treinar.

**Exercício 1.1.** Com que volume mensal a GPU própria passa a ser mais barata do que o Haiku? E do que o Sonnet? Não te esqueças do custo de engenharia e manutenção.

## 2. LoRA à mão

Uma camada linear congelada mais uma correção `B·A` de característica `r`. Como `B` começa a zeros, o modelo inicial é exatamente o original.

In [ ]:
import torch.nn as nn

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False                      # pesos originais congelados
        self.A = nn.Parameter(torch.randn(r, base.in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(base.out_features, r))   # zeros: começa sem alterar nada
        self.escala = alpha / r

    def forward(self, x):
        return self.base(x) + (x @ self.A.T @ self.B.T) * self.escala

base = nn.Linear(1536, 1536)
lora = LoRALinear(base, r=16)
x = torch.randn(4, 1536)
print("Igual ao original no início:", torch.allclose(base(x), lora(x)))
treinaveis = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print(f"Treináveis: {treinaveis:,} de {treinaveis + sum(p.numel() for p in base.parameters()):,} ({treinaveis / sum(p.numel() for p in lora.parameters()):.1%})")

Treinamos só `A` e `B` para imitar uma camada «alvo» que difere da base por uma correção de característica baixa:

In [ ]:
alvo = nn.Linear(1536, 1536)
with torch.no_grad():
    alvo.weight.copy_(base.weight + 0.05 * torch.randn(1536, 4) @ torch.randn(4, 1536))
    alvo.bias.copy_(base.bias)
opt = torch.optim.Adam([lora.A, lora.B], lr=1e-3)
for passo in range(301):
    x = torch.randn(64, 1536)
    perda = ((lora(x) - alvo(x)) ** 2).mean()
    opt.zero_grad(); perda.backward(); opt.step()
    if passo % 100 == 0:
        print(f"passo {passo}: perda {perda.item():.5f}")

**Exercício 2.1.** Repete com `r=2` e `r=64`. O que acontece quando `r` é menor do que a característica real da diferença (4)?

## 3. Chat template e máscara da perda

Vemos o texto que o modelo recebe de facto e que parte conta para a perda. Só precisa do tokenizador (download pequeno).

In [ ]:
from transformers import AutoTokenizer
from comum import exemplo_chat

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
ex = exemplo_chat("Encomenda 48213 chegou com a taça partida!! Quero o dinheiro de volta",
                  {"categoria": "produto_danificado", "urgencia": "media", "numero_encomenda": "48213", "produto": "taça", "pedido": "reembolso"})
prompt_txt = tok.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True)
completo = tok.apply_chat_template(ex["prompt"] + ex["completion"], tokenize=False)
print(completo)
n_prompt, n_total = len(tok(prompt_txt)["input_ids"]), len(tok(completo)["input_ids"])
print(f"\n{n_total} tokens; só os últimos {n_total - n_prompt} (a resposta) contam para a perda.")

Se treinasses com o template de outro modelo (por exemplo, o do Llama com o Qwen), o modelo aprenderia marcadores que nunca vê em produção. Deixa sempre o tokenizador aplicar o template.

## 4. Gerar dados por destilação

O Claude gera reclamações variadas e rotula-as. Para experimentar usa `--n 300` (cerca de um minuto); para treinar a sério, 1500. Custa poucos dólares.

In [ ]:
correr("gerar_dados.py", "--n", "1500")

In [ ]:
from collections import Counter
treino = [json.loads(l) for l in (UC / "dados/treino_bruto.jsonl").read_text(encoding="utf-8").splitlines()]
for x in treino[:5]:
    print(f"- {x['texto'][:120]}\n  → {x['rotulo']}")
print("\nUrgência:", Counter(x["rotulo"]["urgencia"] for x in treino))
print("Pedido:", Counter(x["rotulo"]["pedido"] for x in treino))

**Exercício 4.1.** Lê 20 exemplos ao acaso. Concordas com os rótulos? Os erros do professor passam para o aluno: num projeto real, uma pessoa revê pelo menos o conjunto de teste.

## 5. Linhas de base

Antes de treinar, medimos o modelo pequeno sem treino (com e sem exemplos no prompt) e um modelo grande na API.

In [ ]:
correr("avaliar.py", "--modelo", "base")
correr("avaliar.py", "--modelo", "base", "--few-shot", "3")
correr("avaliar.py", "--modelo", "claude:claude-haiku-5-5")

## 6. SFT com QLoRA

Duas épocas sobre ~1200 exemplos demoram 15 a 25 minutos numa T4.

In [ ]:
correr("treinar_sft.py")

In [ ]:
import matplotlib.pyplot as plt
estado = json.loads(sorted((UC / "saidas/sft").glob("checkpoint-*/trainer_state.json"))[-1].read_text())
tr = [(h["step"], h["loss"]) for h in estado["log_history"] if "loss" in h]
ev = [(h["step"], h["eval_loss"]) for h in estado["log_history"] if "eval_loss" in h]
plt.plot(*zip(*tr), label="treino"); plt.plot(*zip(*ev), "o-", label="validação")
plt.xlabel("passo"); plt.ylabel("perda"); plt.legend(); plt.title("SFT"); plt.show()

In [ ]:
correr("avaliar.py", "--modelo", "saidas/sft/final")

In [ ]:
erros = [json.loads(l) for l in (UC / "resultados/saidas_sft_final_erros.jsonl").read_text(encoding="utf-8").splitlines() if l]
for e in erros[:6]:
    print(f"- {e['texto'][:110]}\n  esperado: {e['esperado']}\n  obtido:   {e['obtido'][:160]}\n")

**Exercício 6.1.** Classifica os erros: o modelo errou, ou o rótulo é discutível? Que campo falha mais? Isso sugere mais dados de que tipo?

## 7. DPO com os erros do próprio modelo

In [ ]:
correr("treinar_dpo.py", "--sft", "saidas/sft/final")
correr("avaliar.py", "--modelo", "saidas/dpo/final")

Compara com o SFT. O DPO costuma ajudar pouco quando o SFT já está perto do teto, e pode piorar se os pares forem ruidosos (rótulos errados como «escolhida»). Mede sempre.

## 8. GRPO: uma recompensa verificável

Com GRPO não precisas de pares, só de uma função que pontue cada resposta. Escrever e testar a recompensa é a parte importante e corre em CPU.

In [ ]:
from comum import ler_json, normalizar, CAMPOS

def recompensa(completions, rotulo, **_):
    """Uma pontuação por resposta: 0 se o JSON é inválido, senão a fração de campos certos (+0,2 de bónus se tudo certo)."""
    pontos = []
    for c, r in zip(completions, rotulo):
        texto = c[0]["content"] if isinstance(c, list) else c
        p = ler_json(texto)
        if p is None:
            pontos.append(0.0)
            continue
        certos = sum(normalizar(p[k]) == normalizar(r[k]) for k in CAMPOS) / len(CAMPOS)
        pontos.append(certos + (0.2 if certos == 1 else 0))
    return pontos

ref = treino[0]["rotulo"]
testes = [json.dumps(ref, ensure_ascii=False), json.dumps({**ref, "urgencia": "alta" if ref["urgencia"] != "alta" else "baixa"}), "não sei"]
print(recompensa(testes, [ref] * 3))

Com o TRL, o treino seria (exercício para quem tem GPU com tempo):

```python
from trl import GRPOConfig, GRPOTrainer
ds = Dataset.from_list([{**exemplo_chat(x["texto"]), "rotulo": x["rotulo"]} for x in treino])
cfg = GRPOConfig(output_dir="saidas/grpo", num_generations=8, max_completion_length=128, learning_rate=1e-6, beta=0.04)
GRPOTrainer(model=modelo_sft, reward_funcs=recompensa, args=cfg, train_dataset=ds, processing_class=tok).train()
```

**Exercício 8.1.** Que «batota» poderia o modelo encontrar nesta recompensa (reward hacking)? E se a recompensa só verificasse o JSON válido?

## 9. Esquecimento, fusão e publicação

In [ ]:
from modelo_local import carregar, gerar
perguntas = [[{"role": "user", "content": "Explica em três frases o que é a inflação."}],
             [{"role": "user", "content": "Escreve um email curto a pedir desculpa a um cliente pelo atraso."}]]
for nome, adaptador in [("base", None), ("sft", str(UC / "saidas/sft/final"))]:
    m, t = carregar(adaptador=adaptador)
    for p, s in zip(perguntas, gerar(m, t, perguntas, max_new_tokens=150)):
        print(f"[{nome}] {p[0]['content']}\n{s.strip()[:400]}\n")
    del m; torch.cuda.empty_cache()

O modelo afinado provavelmente responde em JSON a tudo: aprendeu a tarefa e esqueceu o resto. Para um classificador dedicado é aceitável; para um assistente geral, mistura dados gerais no treino.

Por fim, fundimos o melhor adaptador no modelo base para o servir no módulo 12:

In [ ]:
correr("fundir.py", "--adaptador", "saidas/sft/final")
correr("avaliar.py", "--tabela")

Para publicar: `huggingface-cli login` e `python fundir.py --adaptador saidas/sft/final --hub o-teu-utilizador/nome-do-modelo`. Revê o model card gerado antes.

## Use case: modelo pequeno especialista em português

```bash
cd modulos/11-fine-tuning-e-alinhamento/use-case
uv run python gerar_dados.py --n 1500
uv run python avaliar.py --modelo base && uv run python avaliar.py --modelo claude:claude-haiku-5-5
uv run python treinar_sft.py
uv run python avaliar.py --modelo saidas/sft/final
uv run python treinar_dpo.py && uv run python avaliar.py --modelo saidas/dpo/final
uv run python fundir.py --adaptador saidas/sft/final
uv run python avaliar.py --tabela
```

**Desafio:** repete com `Qwen/Qwen2.5-0.5B-Instruct` e `Qwen/Qwen2.5-3B-Instruct` (`--base`) e desenha a curva qualidade vs tamanho vs latência. Qual é o modelo mais pequeno que chega a 95 % da exatidão do Claude?